# 03 Sign Event → State Machine Contract v1

이 노트북은 YOLO sign detector를 새로 평가하지 않는다. 20-02에서 만든 sign event trigger를 받아서, 최종 runtime의 state machine이 어떤 방식으로 해석할지 정리한다.

핵심 원칙은 다음과 같다.

```text
sign detector/event trigger:
  class, conf, bbox, area_ratio로 one-shot event를 만든다.

lane fixed-base controller:
  steer_lane, mode, center_error를 만든다.

state machine:
  event와 lane 상태를 보고 최종 steer/speed/action을 결정한다.
```

즉 state machine은 08a의 geometry를 직접 고치지 않는다. lane controller가 낸 `steer_norm` 위에서, 필요한 순간에 steer/speed/action을 override한다.

## 1. 왜 sign state machine이 필요한가

YOLO는 표지판을 검출할 뿐이다. `left`, `right`, `straight` 같은 클래스 이름은 차량 제어 명령이 아니다.

예를 들어 `right` 표지판이 검출되면 다음이 필요하다.

```text
1. 지금 검출된 right가 충분히 가까운가?
2. 곧바로 우회전할 것인가, pending turn으로 저장할 것인가?
3. 회전 중 lane detector 출력은 무시할 것인가?
4. 언제 lane follower로 복귀할 것인가?
```

20-02는 1번까지 담당한다. 이 노트북은 2~4번의 계약을 정한다.

In [ ]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path(r"~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization")
EXP_ROOT = PROJECT_ROOT / "10_experiments" / "20_sign_event_contract"
REVIEW_ROOT = EXP_ROOT / "review_outputs" / "03_sign_state_machine_contract"
POLICY_02_JSON = EXP_ROOT / "review_outputs" / "02_sign_event_policy_candidate" / "sign_event_policy_candidate_v1.json"

CONFIG_DIR = REVIEW_ROOT / "config"
TABLE_DIR = REVIEW_ROOT / "tables"
DOC_DIR = REVIEW_ROOT / "docs"
for d in [CONFIG_DIR, TABLE_DIR, DOC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("20-02 policy:", POLICY_02_JSON)
print("review root:", REVIEW_ROOT)

In [ ]:
def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")


def write_text(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")

assert POLICY_02_JSON.exists(), f"Run 20-02 first: {POLICY_02_JSON}"
sign_policy = read_json(POLICY_02_JSON)
print(sign_policy["name"])
display(pd.DataFrame(sign_policy["event_policy"]).T[["event_name", "priority", "hold_sec", "cooldown_sec", "min_conf", "min_area_ratio", "state_machine_hint"]])

## 2. Runtime interface

최종 패키지에서는 세 종류의 입력을 state machine에 넘긴다.

### Lane input

08b fixed-base controller가 만든 값이다.

```python
lane = {
    "steer_norm": -0.18,
    "mode": "both" | "single" | "lost",
    "center_error": 0.04,
    "local_slope": -0.12,
}
```

### Sign event input

20-02 event trigger가 만든 one-shot event다.

```python
sign_event = {
    "class_name": "left",
    "event_name": "sign_left",
    "confidence": 0.82,
    "area_ratio": 0.010,
    "priority": 70,
}
```

### State output

state machine은 최종 drive command를 반환한다.

```python
drive_cmd = {
    "steer_norm": -0.55,
    "speed_scale": 0.65,
    "action": "drive" | "stop" | "horn",
    "reason": "turn_left_override",
}
```

In [ ]:
SIGN_STATE_CONFIG = {
    "name": "sign_state_machine_contract_v1",
    "input_contracts": {
        "sign_event_policy": str(POLICY_02_JSON),
        "lane_controller": "12_clrkdnet_supervised_rebuild/08b_fixed_base_steering_v1",
    },
    "sign_convention": {
        "steer_positive": "image-right before motor steer_sign; negative means image-left",
        "state_machine_scope": "modifies final steer_norm/speed/action, not lane geometry features",
    },
    "default_drive": {
        "speed_scale": 1.0,
        "action": "drive",
    },
    "turn_override": {
        "left_steer": -0.55,
        "right_steer": 0.55,
        "speed_scale": 0.65,
        "min_turn_sec": 0.7,
        "max_turn_sec": 2.8,
        "recover_both_frames": 3,
        "recover_center_error_max": 0.25,
        "after_max_turn_state": "lane_recover",
    },
    "lane_recover": {
        "speed_scale": 0.75,
        "max_sec": 1.5,
        "exit_both_frames": 2,
        "exit_center_error_max": 0.30,
    },
    "straight_bias": {
        "hold_sec": 3.0,
        "max_abs_steer": 0.20,
        "speed_scale": 0.90,
        "meaning": "straight sign does not force a turn; it limits excessive lane steering near straight intersection paths",
    },
    "speed_20": {
        "hold_sec": 5.0,
        "speed_scale": 0.65,
    },
    "stop": {
        "hold_sec": 2.0,
        "steer_norm": 0.0,
        "speed_scale": 0.0,
    },
    "horn": {
        "pulse_sec": 0.5,
        "does_not_change_steer": True,
        "does_not_change_speed": True,
    },
    "priority_order": ["stop", "left", "right", "straight", "speed_20", "horn"],
}

write_json(CONFIG_DIR / "sign_state_machine_contract_v1.json", SIGN_STATE_CONFIG)
print("saved:", CONFIG_DIR / "sign_state_machine_contract_v1.json")
display(pd.Series({
    "turn_left_steer": SIGN_STATE_CONFIG["turn_override"]["left_steer"],
    "turn_right_steer": SIGN_STATE_CONFIG["turn_override"]["right_steer"],
    "turn_speed_scale": SIGN_STATE_CONFIG["turn_override"]["speed_scale"],
    "straight_max_abs_steer": SIGN_STATE_CONFIG["straight_bias"]["max_abs_steer"],
    "speed_20_scale": SIGN_STATE_CONFIG["speed_20"]["speed_scale"],
    "stop_hold_sec": SIGN_STATE_CONFIG["stop"]["hold_sec"],
}).to_frame("value"))

## 3. Class별 action table

각 sign class는 runtime에서 서로 다른 종류의 제어를 만든다.

- `left/right`: lane steering을 잠깐 장악한다.
- `straight`: 큰 좌우 조향을 제한한다.
- `speed_20`: 속도만 낮춘다.
- `stop`: 정지한다.
- `horn`: horn action만 만든다.

In [ ]:
def build_action_table(sign_policy, cfg):
    rows = []
    for class_name, p in sign_policy["event_policy"].items():
        if class_name in ["left", "right"]:
            action_type = "turn_override"
            state_effect = "enter TURN_OVERRIDE; ignore lane steer until minimum turn time, then recover on stable both lane"
            steer_effect = cfg["turn_override"][f"{class_name}_steer"]
            speed_effect = cfg["turn_override"]["speed_scale"]
            exit_rule = f"after {cfg['turn_override']['min_turn_sec']}s and both lane stable {cfg['turn_override']['recover_both_frames']} frames with |center_error| < {cfg['turn_override']['recover_center_error_max']}"
        elif class_name == "straight":
            action_type = "straight_bias"
            state_effect = "limit excessive lane steering for hold_sec"
            steer_effect = f"clamp to +/-{cfg['straight_bias']['max_abs_steer']}"
            speed_effect = cfg["straight_bias"]["speed_scale"]
            exit_rule = f"hold {cfg['straight_bias']['hold_sec']}s or superseded by higher priority event"
        elif class_name == "speed_20":
            action_type = "speed_scale"
            state_effect = "slow driving mode"
            steer_effect = "unchanged"
            speed_effect = cfg["speed_20"]["speed_scale"]
            exit_rule = f"hold {cfg['speed_20']['hold_sec']}s"
        elif class_name == "stop":
            action_type = "stop_hold"
            state_effect = "stop motors"
            steer_effect = cfg["stop"]["steer_norm"]
            speed_effect = cfg["stop"]["speed_scale"]
            exit_rule = f"hold {cfg['stop']['hold_sec']}s"
        elif class_name == "horn":
            action_type = "horn_pulse"
            state_effect = "one-shot horn pulse"
            steer_effect = "unchanged"
            speed_effect = "unchanged"
            exit_rule = f"pulse {cfg['horn']['pulse_sec']}s"
        else:
            action_type = "unknown"
            state_effect = "ignored"
            steer_effect = "unchanged"
            speed_effect = "unchanged"
            exit_rule = "none"
        rows.append({
            "class_name": class_name,
            "event_name": p["event_name"],
            "event_priority": p["priority"],
            "trigger_hold_sec": p["hold_sec"],
            "action_type": action_type,
            "state_effect": state_effect,
            "steer_effect": steer_effect,
            "speed_effect": speed_effect,
            "exit_rule": exit_rule,
        })
    return pd.DataFrame(rows).sort_values(["event_priority", "class_name"], ascending=[False, True])

action_table = build_action_table(sign_policy, SIGN_STATE_CONFIG)
action_table.to_csv(TABLE_DIR / "sign_state_action_table.csv", index=False, encoding="utf-8-sig")
display(action_table)
print("saved:", TABLE_DIR / "sign_state_action_table.csv")

## 4. Reference state machine pseudo-code

아래 코드는 최종 runtime 코드가 아니라, 구현 기준을 고정하기 위한 reference다.

핵심 상태는 네 개다.

```text
LANE_FOLLOW:
  기본 lane controller 사용

TURN_OVERRIDE:
  left/right 표지판 이후 수동 조향 장악

LANE_RECOVER:
  turn 시간이 끝났지만 both lane 복귀가 불안정할 때 저속 lane follow

STOP_HOLD:
  stop 표지판 처리
```

In [ ]:
def initial_state():
    return {
        "state": "LANE_FOLLOW",
        "state_time": 0.0,
        "pending_turn": None,
        "stable_both_count": 0,
        "straight_until": 0.0,
        "slow_until": 0.0,
        "stop_until": 0.0,
        "horn_until": 0.0,
        "last_reason": "init",
    }


def choose_event(events):
    if not events:
        return None
    priority = {name: i for i, name in enumerate(reversed(SIGN_STATE_CONFIG["priority_order"]))}
    return max(events, key=lambda e: (sign_policy["event_policy"].get(e["class_name"], {}).get("priority", 0), e.get("confidence", 0.0)))


def update_stable_both_count(state, lane, cfg):
    if lane.get("mode") == "both" and abs(float(lane.get("center_error", 999.0))) <= cfg["turn_override"]["recover_center_error_max"]:
        state["stable_both_count"] += 1
    else:
        state["stable_both_count"] = 0


def apply_timed_effects(cmd, t, state, cfg):
    if t < state["straight_until"]:
        m = cfg["straight_bias"]["max_abs_steer"]
        cmd["steer_norm"] = float(np.clip(cmd["steer_norm"], -m, m))
        cmd["speed_scale"] = min(cmd["speed_scale"], cfg["straight_bias"]["speed_scale"])
        cmd["reason"] += "+straight_clamp"
    if t < state["slow_until"]:
        cmd["speed_scale"] = min(cmd["speed_scale"], cfg["speed_20"]["speed_scale"])
        cmd["reason"] += "+speed_20"
    if t < state["horn_until"]:
        cmd["action"] = "horn"
        cmd["reason"] += "+horn"
    return cmd


def update_state_machine(lane, events, state, cfg, dt=0.1, t=0.0):
    state = dict(state)
    state["state_time"] += dt
    event = choose_event(events)

    if event is not None:
        c = event["class_name"]
        if c == "stop":
            state["state"] = "STOP_HOLD"
            state["state_time"] = 0.0
            state["stop_until"] = t + cfg["stop"]["hold_sec"]
            state["last_reason"] = "event_stop"
        elif c in ["left", "right"]:
            state["state"] = "TURN_OVERRIDE"
            state["state_time"] = 0.0
            state["pending_turn"] = c
            state["stable_both_count"] = 0
            state["last_reason"] = f"event_turn_{c}"
        elif c == "straight":
            state["straight_until"] = max(state["straight_until"], t + cfg["straight_bias"]["hold_sec"])
            state["last_reason"] = "event_straight"
        elif c == "speed_20":
            state["slow_until"] = max(state["slow_until"], t + cfg["speed_20"]["hold_sec"])
            state["last_reason"] = "event_speed_20"
        elif c == "horn":
            state["horn_until"] = max(state["horn_until"], t + cfg["horn"]["pulse_sec"])
            state["last_reason"] = "event_horn"

    cmd = {
        "steer_norm": float(lane.get("steer_norm", 0.0)),
        "speed_scale": cfg["default_drive"]["speed_scale"],
        "action": cfg["default_drive"]["action"],
        "reason": "lane_follow",
    }

    if state["state"] == "STOP_HOLD":
        cmd.update({
            "steer_norm": cfg["stop"]["steer_norm"],
            "speed_scale": cfg["stop"]["speed_scale"],
            "action": "stop",
            "reason": "stop_hold",
        })
        if t >= state["stop_until"]:
            state["state"] = "LANE_FOLLOW"
            state["state_time"] = 0.0
            state["last_reason"] = "stop_done"

    elif state["state"] == "TURN_OVERRIDE":
        turn = state["pending_turn"] or "right"
        steer = cfg["turn_override"][f"{turn}_steer"]
        cmd.update({
            "steer_norm": steer,
            "speed_scale": cfg["turn_override"]["speed_scale"],
            "action": "drive",
            "reason": f"turn_{turn}_override",
        })
        update_stable_both_count(state, lane, cfg)
        can_recover = (
            state["state_time"] >= cfg["turn_override"]["min_turn_sec"]
            and state["stable_both_count"] >= cfg["turn_override"]["recover_both_frames"]
        )
        timed_out = state["state_time"] >= cfg["turn_override"]["max_turn_sec"]
        if can_recover:
            state["state"] = "LANE_FOLLOW"
            state["state_time"] = 0.0
            state["pending_turn"] = None
            state["last_reason"] = "turn_recovered_by_both_lane"
        elif timed_out:
            state["state"] = "LANE_RECOVER"
            state["state_time"] = 0.0
            state["last_reason"] = "turn_timeout_lane_recover"

    elif state["state"] == "LANE_RECOVER":
        cmd["speed_scale"] = min(cmd["speed_scale"], cfg["lane_recover"]["speed_scale"])
        cmd["reason"] = "lane_recover"
        update_stable_both_count(state, lane, cfg)
        if state["stable_both_count"] >= cfg["lane_recover"]["exit_both_frames"]:
            state["state"] = "LANE_FOLLOW"
            state["state_time"] = 0.0
            state["pending_turn"] = None
            state["last_reason"] = "recover_done"
        elif state["state_time"] >= cfg["lane_recover"]["max_sec"]:
            state["state"] = "LANE_FOLLOW"
            state["state_time"] = 0.0
            state["pending_turn"] = None
            state["last_reason"] = "recover_timeout"

    if state["state"] == "LANE_FOLLOW":
        cmd = apply_timed_effects(cmd, t, state, cfg)

    cmd["steer_norm"] = float(np.clip(cmd["steer_norm"], -1.0, 1.0))
    cmd["speed_scale"] = float(np.clip(cmd["speed_scale"], 0.0, 1.0))
    return cmd, state

print("reference state machine functions ready")

## 5. Synthetic trace sanity check

실제 map sequence가 없으므로, 여기서는 짧은 synthetic sequence로 상태 전이가 의도대로 되는지만 확인한다.

목표:

```text
right event → TURN_OVERRIDE → both lane 안정 검출 → LANE_FOLLOW 복귀
speed_20 event → speed_scale 감소
straight event → steer clamp
stop event → STOP_HOLD
```

In [ ]:
def make_lane(t, mode="both", steer=0.0, center_error=0.0):
    return {
        "mode": mode,
        "steer_norm": steer,
        "center_error": center_error,
        "local_slope": 0.0,
    }


def event(class_name, conf=0.8):
    return {"class_name": class_name, "confidence": conf, "event_name": f"sign_{class_name}"}

trace_rows = []
state = initial_state()
dt = 0.1
for frame in range(90):
    t = frame * dt
    events = []
    lane = make_lane(t, mode="single", steer=0.10, center_error=0.0)

    if frame == 5:
        events = [event("speed_20")]
    if frame == 15:
        events = [event("right")]
    if 35 <= frame <= 40:
        lane = make_lane(t, mode="both", steer=0.03, center_error=0.08)
    if frame == 50:
        events = [event("straight")]
    if 50 <= frame <= 65:
        lane = make_lane(t, mode="both", steer=0.45, center_error=0.02)
    if frame == 70:
        events = [event("stop")]

    cmd, state = update_state_machine(lane, events, state, SIGN_STATE_CONFIG, dt=dt, t=t)
    trace_rows.append({
        "frame": frame,
        "t": round(t, 2),
        "event": events[0]["class_name"] if events else "",
        "lane_mode": lane["mode"],
        "lane_steer": lane["steer_norm"],
        "lane_center_error": lane["center_error"],
        "state": state["state"],
        "pending_turn": state["pending_turn"] or "",
        "stable_both_count": state["stable_both_count"],
        "cmd_steer": cmd["steer_norm"],
        "cmd_speed_scale": cmd["speed_scale"],
        "cmd_action": cmd["action"],
        "cmd_reason": cmd["reason"],
        "state_reason": state["last_reason"],
    })

trace_df = pd.DataFrame(trace_rows)
trace_df.to_csv(TABLE_DIR / "synthetic_sign_state_trace.csv", index=False, encoding="utf-8-sig")
display(trace_df.iloc[[0,5,15,20,35,38,50,55,70,75,89]])
print("saved:", TABLE_DIR / "synthetic_sign_state_trace.csv")

In [ ]:
pseudocode_md = r'''
# Sign State Machine v1 - Reference Pseudocode

이 문서는 최종 runtime 구현 시 따라야 할 sign event 처리 기준이다.

## Inputs

```python
lane = {
    "steer_norm": float,
    "mode": "both" | "single" | "lost",
    "center_error": float,
    "local_slope": float,
}

sign_events = [
    {"class_name": "left", "confidence": 0.8, "area_ratio": 0.01, ...}
]
```

## Outputs

```python
drive_cmd = {
    "steer_norm": float,
    "speed_scale": float,
    "action": "drive" | "stop" | "horn",
    "reason": str,
}
```

## Rules

- `left/right`: enter `TURN_OVERRIDE`; use fixed steer and reduced speed. Exit only after minimum turn time and stable both-lane recovery.
- `straight`: do not force steer to zero. Clamp excessive lane steer for a short hold window.
- `speed_20`: reduce speed only.
- `stop`: stop for fixed hold time.
- `horn`: emit one-shot horn action, without changing steer/speed.

## Recovery Rule

A left/right turn returns to lane following when:

```text
state_time >= min_turn_sec
and lane.mode == both for recover_both_frames
and abs(center_error) < recover_center_error_max
```

If this does not happen before `max_turn_sec`, enter `LANE_RECOVER` and use low-speed lane following.
'''
write_text(DOC_DIR / "sign_state_machine_reference_pseudocode.md", pseudocode_md)
print("saved:", DOC_DIR / "sign_state_machine_reference_pseudocode.md")

In [ ]:
report = {
    "name": "03_sign_state_machine_contract_v1",
    "source_policy": str(POLICY_02_JSON),
    "purpose": "Define how YOLO sign trigger events are consumed by the final drive state machine.",
    "contract": SIGN_STATE_CONFIG,
    "action_table_csv": str(TABLE_DIR / "sign_state_action_table.csv"),
    "synthetic_trace_csv": str(TABLE_DIR / "synthetic_sign_state_trace.csv"),
    "pseudocode_md": str(DOC_DIR / "sign_state_machine_reference_pseudocode.md"),
    "implementation_notes": [
        "State machine does not modify lane geometry features directly.",
        "Left/right override final steer_norm and speed_scale temporarily.",
        "Return from turn override is gated by stable both-lane recovery.",
        "Straight sign clamps excessive steer instead of forcing zero steer.",
        "This notebook is a design contract; live timing must be tuned on Pi."
    ],
}
write_json(REVIEW_ROOT / "sign_state_machine_contract_report_v1.json", report)
print("saved:", REVIEW_ROOT / "sign_state_machine_contract_report_v1.json")
print(json.dumps({
    "contract": str(CONFIG_DIR / "sign_state_machine_contract_v1.json"),
    "action_table": str(TABLE_DIR / "sign_state_action_table.csv"),
    "trace": str(TABLE_DIR / "synthetic_sign_state_trace.csv"),
}, indent=2, ensure_ascii=False))

## 6. 다음 단계

20번 sign 흐름은 여기까지로 닫을 수 있다.

```text
20-00: YOLO 출력 형태 확인
20-01: detection gate 확인
20-02: sign event trigger 후보
20-03: sign event를 state machine이 소비하는 방식 정의
```

다음 작업은 color event도 같은 방식으로 정리한 뒤, 최종 runtime package에서 다음 순서로 결합하는 것이다.

```text
lane fixed-base steer
+ sign state machine
+ traffic/redline state machine
→ final motor command
```